In [ ]:
import sys
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.getcwd())

from src.models import CNN
from src.train import train
from src.dataset import HAM10000Dataset, get_train_transforms, get_eval_transforms, compute_class_weights, IMAGENET_MEAN, IMAGENET_STD, DX_TO_IDX, IDX_TO_DX
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

2.12.1+cu130
13.0


Read in training and validation data, initialize HAM10000Dataset objects for each

In [4]:
train_df = pd.read_csv('data/splits/train.csv')
val_df = pd.read_csv('data/splits/val.csv')

image_dir = os.path.join(os.getcwd(), 'data', 'HAM10000_images')

train_dataset = HAM10000Dataset(train_df, image_dir, get_train_transforms())
val_dataset = HAM10000Dataset(val_df, image_dir, get_eval_transforms())

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2)

Create CNN

In [5]:
cnn = CNN(
   in_channels=3,
   num_classes=7,
   n_blocks=4,
   n_layers_per_block=6,
   channels_l0=16,
   stem_kernel_size=11,
   stem_stride=2,
   block_stride=2,
   channel_growth_factor=2,
   dropout_p=0.3 
)

Construct loss function

In [6]:
class_weights = compute_class_weights(train_df)

loss_func = torch.nn.CrossEntropyLoss(weight=class_weights)

Construct Optimizer

In [7]:
lr = 1e-3
momentum = 0.9
adaptive_learning_rate = 0.999
betas = (momentum, adaptive_learning_rate)
optimizer = torch.optim.AdamW(cnn.parameters(), lr=lr, betas=betas)

Train CNN

In [8]:
cnn = train(
    model=cnn,
    model_name="baseline_cnn",
    train_loader=train_loader,
    val_loader=val_loader,
    loss_func=loss_func,
    optimizer=optimizer,
    num_epoch=50
)

Using CUDA
Epoch  1 / 50: train_acc=0.2507 val_acc=0.2952 train_f1=0.1391 val_f1=0.1466 
Epoch 10 / 50: train_acc=0.4460 val_acc=0.4785 train_f1=0.2660 val_f1=0.2707 
Epoch 20 / 50: train_acc=0.5371 val_acc=0.5363 train_f1=0.3598 val_f1=0.3606 
Epoch 30 / 50: train_acc=0.5743 val_acc=0.6383 train_f1=0.3986 val_f1=0.4427 
Epoch 40 / 50: train_acc=0.5992 val_acc=0.6098 train_f1=0.4369 val_f1=0.4512 
Epoch 50 / 50: train_acc=0.6082 val_acc=0.5588 train_f1=0.4578 val_f1=0.4322 
Model saved to logs\baseline_cnn_0707_161948\baseline_cnn.th
